<a href="https://colab.research.google.com/github/minjoonkim01/aat3020/blob/2025/NLP_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1

In this assignment, you will explore about word vectors.

- Submision: A report in ``pdf``, your completed notebook file in ``ipynb``, and training data in ``txt``
    - The assignment will be evalulated mainly with report. So please include every detail you want to present in your report, including figures.
    - Report: Free format. You can copy and paste part of your code for some problems.
      - Report has to be written in English
    - ipynb: Save your notebook (with output of each cell if possible) as ipynb and submit it
- Evaluation criteria
    - How interesting and original are the presented examples
    - How well you describe the reason of success or failure of your examples by considering how Word2Vec is trained
    - Any description that is suspicious for using LLM without understanding the content can be penalized. You may use LLM for translation, but you have to describe it in your report.

## 0. Setup
- Check ``gensim`` library is installed
  - if not, you can install using ``!pip install gensim``
- List the downloadable vectors from ``gensim``


In [ ]:
import gensim
import numpy as np
import pprint as pp

In [ ]:
import gensim.downloader
list(gensim.downloader.info()['models'].keys())

- Among the Word2Vec model codes above, select one model of your choice among ``glove-wiki-gigaword`` or ``glove-twitter``
    - numbers at the last represents the number of dimension of each Word2Vec Model
        - e.g. ``glove-twitter-200`` was trained on twitter dataset while embedding each word into 200-dim vector
        - e.g. ``glove-wiki-gigaword-300`` was trained on wikipedia dataset while embedding each word into 300-dim vector
- Download the selected model and load it as a ``model``

In [ ]:
your_model_code = 'glove-wiki-gigaword-300' # select among the model code aboves
model = gensim.downloader.load(your_model_code) # download and load the model. It can take some time

In [ ]:
# test the model output
model['cat']

## Problem 1. Simple Mathematics with Word2Vec
- In this problem, you have to complete the given functions ``word_analogy_with_vector`` and ``get_cosine_similarity``
  - To get the exactly same result with ``model.most_similar()``, you have to normalize each vector before doring arithmetic.
  - Using L2 norm (sqrt of sum of square of every item in the vector)
  - The result will also naturally include the positive query words itsef.
- In your report, **please include your code for these functions**


In [ ]:
def word_analogy_with_vector(model, x_1, x_2, y_1):
  '''
  This function takes a gensim Word2Vec model and outputs a vector to find y2 that corresponds to x_1 → x_2 == y_1 → y_2
  e.g. x_1 (man) → x_2 (king) == y_1 (woman) → y_2(?)

  inputs
  model (gensim.models.keyedvectors.KeyedVectors): Word2Vec model in KeyedVectors in gensim library
  x_1, x_2, y_1 (str): Words in the model's vocabulary.

  output (np.ndarray): A vector in np.ndarray, which can be used to find proper y_2 for given (model, x_1, x_2, y_1)
  
  CAUTION: You have to normalize (divide vector by its length) the vector before doing arithmetic.
  '''

  # Write your code from here
  # FIXME(mjkim)
  x_1_vec = model[x_1]
  x_2_vec = model[x_2]
  y_1_vec = model[y_1]
  x_1_vec = x_1_vec / np.linalg.norm(x_1_vec)
  x_2_vec = x_2_vec / np.linalg.norm(x_2_vec)
  y_1_vec = y_1_vec / np.linalg.norm(y_1_vec)
  y_2_vec = x_2_vec - x_1_vec + y_1_vec
  return y_2_vec / np.linalg.norm(y_2_vec)

# test whether the function works well
result_vector = word_analogy_with_vector(model, 'man', 'king', 'woman')
print('result vector is ', result_vector)
assert isinstance(result_vector, np.ndarray), "Output of the function has to be np.ndarray"
model.most_similar(result_vector)

result vector is  [-0.02658517 -0.04160886 -0.00044333  0.07384589  0.07670759 -0.05633707
  0.02913646  0.00121186  0.01681585 -0.11603299 -0.06636501 -0.08482524
  0.0541742   0.07546048  0.02971867  0.01393669  0.05596513  0.04366722
 -0.02761563 -0.03688665 -0.13345417  0.0313828  -0.02542753  0.00880393
  0.02921788 -0.06533846 -0.03727645 -0.04036562  0.05381797  0.01239562
 -0.05869988 -0.04280932 -0.05633385  0.0279524  -0.12134089  0.00106827
 -0.04779797  0.00427856  0.05539053  0.00665951  0.02281769 -0.04314518
 -0.0169343   0.04448815  0.01804601 -0.03271446  0.04605543  0.03789983
  0.00713702 -0.08902845 -0.02792343  0.00888494  0.10250489 -0.08973602
 -0.0545053  -0.02515997  0.04358013  0.00393123  0.07951713  0.01445701
  0.00191055 -0.03545392  0.01051941  0.06738194  0.0346726  -0.11837872
  0.01963535 -0.00024648  0.00871079  0.07197622  0.00356459 -0.06237991
 -0.02680216  0.01144218  0.05100939  0.03233787  0.06609751 -0.08862417
  0.03237421  0.00556114  0.04787

[('king', 0.7572609186172485),
 ('queen', 0.6713277101516724),
 ('princess', 0.5432624220848083),
 ('throne', 0.5386104583740234),
 ('monarch', 0.5347574949264526),
 ('daughter', 0.49802514910697937),
 ('mother', 0.49564430117607117),
 ('elizabeth', 0.4832652509212494),
 ('kingdom', 0.47747084498405457),
 ('prince', 0.4668239951133728)]

In [ ]:
def get_cosine_similarity(model, x, y):
  '''
  This function returns cosine similarity of x,y

  inputs
  model (gensim.models.keyedvectors.KeyedVectors): Word2Vec model in KeyedVectors in gensim library
  x, y (str): Words in the model's vocabulary.

  output
  similarity (float): cosine similarity between x's vector and y's vector
  '''
  # Write your codes from here
  x_vec = model[x]
  y_vec = model[y]
  return np.dot(x_vec, y_vec) / (np.linalg.norm(x_vec) * np.linalg.norm(y_vec))

# test the output with your own choice
word_a = 'good'
word_b = 'bad'

similarity = get_cosine_similarity(model, word_a, word_b)
print(similarity)
assert -1 <= similarity <= 1, "Similarity has to be between -1 and 1"

print('gensim library result:', model.similarity(word_a, word_b))

## Problem 2. Find Most Similar Words
- One of the most simple and typical use case of Word2Vec is finding a word based on similarity.
- You can list the most similar words for a given query word by using ``model.most_similar(your_word)``
    - Usually, every word in Word2Vec model is in lowercase
- **In your report**, present more than **5** interesting examples and explain **why it was interesting for you**
    - Try to explain why those words are regarded similar in Word2Vec, considering how it was trained
- Caution: The model was trained with multilingual dataset. This means the meaning of the word can follow non-English words.
    - e.g. "die", "war" are more frequently used in German than English.

In [21]:
# FIXME(mjkim)
target_word = 'sogang' # Enter your word string here
# check the word is in the vocabulary of the model
assert model.has_index_for(target_word), f"The selected word, {target_word}, is not included in the model's vocabulary"
model.most_similar(target_word)

[('hongik', 0.5292598605155945),
 ('myongji', 0.5273969769477844),
 ('kyungnam', 0.5227178931236267),
 ('panteion', 0.5203900337219238),
 ('kokugakuin', 0.5185405611991882),
 ('yonsei', 0.4931592345237732),
 ('soongsil', 0.47877761721611023),
 ('ewha', 0.47871261835098267),
 ('konkuk', 0.4706515967845917),
 ('dongguk', 0.468786358833313)]

## Problem 3. Word Analogy
- Another interesting thing you can play with Word2Vec is word analogy
- Word analogy is done by adding and subtracting the word vector
- In the cell below, you can run an example like this
    - ``analogy('man', 'king', 'woman')`` represents a question of "man is to king as woman is to what?"
- **Caution**: Do not confuse the relation between each input word.
    - Some wrong examples: 
      - ``analogy(model, 'student', 'school', 'employee')``  Student: School -> Employee: Teacher?
        - This is wrong because the relation between Student and School is not the same as the relation between Employee and Teacher.
      - ``analogy(model, 'android', 'electricity', 'blood')``
        - This is wrong because it calculates ``electricity - android + blood`` instead of ``android - electricity + blood``
- Try with your own choice.
- **In your report**, present at least **5** interesting examples of your choice
    - You can include the failure case
    - Describe what did you expect and why the result was interesting for you

In [22]:
# FIXME(mjkim)
def analogy(model, x1, x2, y1):
  pp.pprint(model.most_similar([x2, y1], negative=[x1]))

# Try with your own word choice
analogy(model, 'man', 'king', 'woman')

[('queen', 0.6713277101516724),
 ('princess', 0.5432624816894531),
 ('throne', 0.5386103987693787),
 ('monarch', 0.5347574949264526),
 ('daughter', 0.49802514910697937),
 ('mother', 0.49564430117607117),
 ('elizabeth', 0.4832652509212494),
 ('kingdom', 0.47747090458869934),
 ('prince', 0.4668239951133728),
 ('wife', 0.46473270654678345)]


## Problem 4. Visualize Word Vectors
- Select a list of words of your interest
    - **At least 30 words for minimum**
    - ``word_list`` is a list of strings
    - every element in ``word_list`` has to be included in the model's vocabulary
- Visualize the vectors of words using dimensionality reduction (in this case, PCA)
- In your report, describe how words are located in 2D space
    - How are the words clustered?
    - Do you think the words are properly located based on their semantic meanings?
    - Is there anything suprising or unexpected examples?

In [ ]:
# Run this cell to
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import plotly.express as px

def display_pca_scatterplot(model, words=None, sample=0):
  if len(words) < 30:
    print("WARNING: For your report, please select more than 30 word samples for the visualization")
    print(f"Current length of input word list: {len(words)}")
  word_vectors = np.array([model[w] for w in words])

  twodim = PCA().fit_transform(word_vectors)[:,:2]

  # plt.figure(figsize=(12,12))
  # plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
  # for word, (x,y) in zip(words, twodim):
  #     plt.text(x+0.05, y+0.05, word, fontsize=15)
  fig = px.scatter(twodim, x=0, y=1, text=words)
  fig.update_traces(textposition='top center')
  fig.show()



In [ ]:
# Select word list of your own interests
word_list = [
    "king",
    "queen",
    "word"
]

display_pca_scatterplot(model, word_list)

## Problem 5. Train New Word2Vec
- Word2Vec models can be trained on different corpus (text)
- Train your own model with your custom selection of text
- In your report, present at least **5** interesting examples that makes different result by dataset selection
    - You can compare some word analogy examples or similairites or visualization
    - You don't have to repeat all the analysis again. Select some examples that you think are interesting
- Explain the difference of the result by dataset selection
- You can refer [Official Documentation](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec) Word2Vec Model

In [ ]:
# You don't have to change this cell
import string
from gensim.models import Word2Vec

def remove_punctuation(x):
  return x.translate(''.maketrans('', '', string.punctuation))
def make_tokenized_corpus(corpus):
  out= [ [y.lower() for y in remove_punctuation(sentence).split(' ') if y] for sentence in corpus]
  return [x for x in out if x!=[]]

In [ ]:
your_text_fn = 'mjkim' # Enter your text file name here

with open(your_text_fn, 'r') as f:
  strings = f.readlines()

'''
This line is for the case when the text file is not properly formatted.
It was used to ignore linebreaks and join the sentences into one string, since the text example included linebreak following printed book lines.

strings = "".join(strings).replace('\n', ' ').replace('Mr.', 'mr').replace('Mrs.', 'mrs').split('. ')
'''
# The strings has to be a list of list of strings, where inner list is a sentence

print("Checking the first 5 sentences in the text file")
for i in range(5):
  print(f"Sentence {i+1}: {strings[i]}")
corpus = make_tokenized_corpus(strings)

- gensim Word2Vec arguments
  - ``sentences``: list of list of strings
  - ``vector_size``: dimension of word vector
  - ``epochs``: number of epoch to train the word2vec model
  - ``window``: maximum distance between the current and predicted word within a sentence
  - ``min_count``: ignore all words with total frequency lower than this
  - ``sg``: training algorithm: 1 for skip-gram; otherwise CBOW
  - ``negative``: if > 0, negative sampling will be used, the int for negative specifies how many "noise words" should be drown (usually between 5-20)

In [ ]:
model = Word2Vec(sentences=corpus, vector_size=200, window=5, min_count=2, epochs=50, sg=1)
model = model.wv # To match with previous codes, we use wv (KeyedVector) of the Word2Vec class
# Try the function above with the newely trained model